# TB4 — Segmentación, cálculos analíticos y fuentes para Tableau

**Proyecto**: Dinámica de precios residenciales en Dinamarca 1992–2024
**Curso**: Data Visualization — UPC 2026-01
**Equipo**: Vilchez · Ballón · Velásquez Borasino
**Entrega**: 4 — Semana sugerida 11

**Pregunta analítica**: ¿Cómo se han diferenciado los precios residenciales entre la
región capital y las provincias danesas bajo distintos regímenes de tasas de interés
e inflación, y qué tipologías de vivienda muestran mayor volatilidad y peores drawdowns
durante las crisis financieras entre 1992 y 2024?

## Contenido de esta entrega

1. Carga de datos (Silver layer)
2. Estructura relacional (Star Schema) para Tableau
3. Validación de integridad referencial y control de duplicidad de métricas
4. Métricas derivadas para visualización (5 KPIs, grano documentado)
5. Segmentación (4 segmentos relevantes para las hipótesis)
6. Parámetros y lógica analítica centralizados
7. Validación de consistencia
8. Exportación de fuentes finales/semidefinitivas para Tableau

> **Nota de reproducibilidad**: este notebook se ejecuta sobre el **Silver layer real**
> generado en TB2 (`danish_housing_clean.parquet`, **1,507,908 filas × 27 columnas**,
> `configs/analysis.yaml -> paths.silver_parquet`). A diferencia de la versión de
> desarrollo (TB4 "smoke test" con muestra sintética), aquí **todos los números,
> tablas y marts provienen del dataset completo de Kaggle (1992–2024) ya limpiado
> y flageado en TB2** — la lógica del notebook es idéntica a la versión de
> desarrollo; sólo cambió la fuente (sección 1) y dos definiciones de segmentación
> que dependían de los valores categóricos sintéticos (sección 2, ver nota).


## 0. Setup y parámetros centralizados

Todos los parámetros analíticos de esta entrega viven en un único diccionario `PARAMS`
(sección 6 desarrolla cada uno). Esto evita "números mágicos" repetidos entre métricas,
y es consistente con el patrón ya usado en `configs/analysis.yaml`.


In [1]:
import sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

# Localizar la raíz del proyecto (donde vive configs/analysis.yaml)
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "configs" / "analysis.yaml").exists():
    if (ROOT / "configs").exists():
        break
    ROOT = ROOT.parent

OUT_MARTS = ROOT / "data" / "processed" / "tableau_marts"
OUT_STAR  = ROOT / "data" / "processed" / "star_schema"
OUT_MARTS.mkdir(parents=True, exist_ok=True)
OUT_STAR.mkdir(parents=True, exist_ok=True)

# ── Parámetros centralizados de TB4 ────────────────────────────────────────
# Mantener sincronizado con configs/analysis.yaml -> kpis / cleaning / marts
PARAMS = {
    # KPI 1: precio real. En el Silver real, sqm_price_real ya viene calculado por
    # TB2 usando la serie real dk_ann_interest_rate_pct (compuesta hasta 2024),
    # NO el deflactor simplificado de 2% anual. cpi_base_year / cpi_annual_rate
    # se conservan como metadato de referencia / fallback documental.
    "cpi_base_year": 2024,
    "cpi_annual_rate": 0.02,

    # KPI 2: índice regional normalizado
    "index_base_year": 1992,

    # Crisis financieras consideradas (GFC + COVID)
    "crisis_years": list(range(2007, 2013)) + [2020, 2021],

    # KPI 3: drawdown — mínimo de observaciones por celda (evita artefactos
    # de muestra mínima)
    "drawdown_min_obs": 5,

    # KPI 4: volatilidad — ventana rolling en trimestres
    "volatility_window_q": 4,

    # KPI 5: elasticidad volumen-bonos — rezago en trimestres
    "macro_lag_quarters": 2,

    # P5 (heredado de TB2): multiplicador IQR para outliers de precio
    "iqr_multiplier": 3.0,
}

print(json.dumps(PARAMS, indent=2))


{
  "cpi_base_year": 2024,
  "cpi_annual_rate": 0.02,
  "index_base_year": 1992,
  "crisis_years": [
    2007,
    2008,
    2009,
    2010,
    2011,
    2012,
    2020,
    2021
  ],
  "drawdown_min_obs": 5,
  "volatility_window_q": 4,
  "macro_lag_quarters": 2,
  "iqr_multiplier": 3.0
}


## 1. Carga de datos (Silver layer real)

Se carga el Silver layer real generado en TB2: `danish_housing_clean.parquet`
(1,507,908 filas × 27 columnas, 1992–2024), con los 8 flags de calidad P1–P8
(`sales_type_valido`, `purchase_price_outlier`, `sqm_price_outlier`,
`periodo_preliminar`, `macro_nulo`, `year_build_flag`, `zip_code` como string de
4 dígitos, columnas macro renombradas `*_pct`) y la métrica `sqm_price_real`
ya deflactada con la serie real de inflación danesa (base 2024).


In [6]:
SILVER_PATH = Path("processed_danish_housing_clean.parquet")

if not SILVER_PATH.exists():
    raise FileNotFoundError(
        f"{SILVER_PATH} no encontrado. Este notebook requiere el Silver layer real "
        "generado por TB2 (scripts/run_cleaning.py)."
    )

silver = pd.read_parquet(SILVER_PATH)
print(f"Fuente: Silver real ({SILVER_PATH})")
print(f"Filas: {len(silver):,} | Columnas: {silver.shape[1]}")
silver.head(3)

Fuente: Silver real (processed_danish_housing_clean.parquet)
Filas: 1,507,908 | Columnas: 27


,date,quarter,house_id,house_type,sales_type,year_build,purchase_price,pct_change_offer_purchase,no_rooms,sqm,sqm_price,address,zip_code,city,area,region,nom_interest_rate_pct,dk_ann_infl_rate_pct,yield_mortgage_bonds_pct,macro_nulo,sales_type_valido,year_build_flag,purchase_price_outlier,sqm_price_outlier,year,periodo_preliminar,sqm_price_real
0,2024-10-26,2024Q4,0,Villa,regular_sale,1974,4350000,0.0,5,215.0,20232.558594,Kildevangen 5,8382,Hinnerup,East & mid jutland,Jutland,3.1,NaN,NaN,True,True,False,False,False,2024,False,20232.56
1,2024-10-26,2024Q4,2,Summerhouse,regular_sale,1956,450000,0.0,3,36.0,12500.000000,Lykkestien 2,4400,Kalundborg,Other islands,Zealand,3.1,NaN,NaN,True,True,False,False,False,2024,False,12500.00
2,2024-10-26,2024Q4,1,Farm,regular_sale,1955,6600000,0.0,3,180.0,36666.667969,Sæderupvej 58,9260,Gistrup,North jutland,Jutland,3.1,NaN,NaN,True,True,False,False,False,2024,False,36666.67


In [ ]:
# Valores categóricos reales (distintos de la muestra sintética de desarrollo):
#  - region:     Bornholm, Fyn & islands, Jutland, Zealand   (agregados geográficos amplios)
#  - house_type: Apartment, Farm, Summerhouse, Townhouse, Villa
#  - area:       8 sub-zonas (incluye 'Capital, Copenhagen', la única que mapea a la capital)
print("region:    ", sorted(silver["region"].unique().tolist()))
print("house_type:", sorted(silver["house_type"].unique().tolist()))
print("area:      ", sorted(silver["area"].unique().tolist()))
print()
print("Filas con sales_type_valido & ~purchase_price_outlier (filtro 'base'):",
      f"{(silver['sales_type_valido'] & ~silver['purchase_price_outlier']).sum():,}",
      f"({(silver['sales_type_valido'] & ~silver['purchase_price_outlier']).mean():.1%})")


region:     ['Bornholm', 'Fyn & islands', 'Jutland', 'Zealand']
house_type: ['Apartment', 'Farm', 'Summerhouse', 'Townhouse', 'Villa']
area:       ['Bornholm', 'Capital, Copenhagen', 'East & mid jutland', 'Fyn & islands', 'North Zealand', 'North jutland', 'Other islands', 'South jutland']

Filas con sales_type_valido & ~purchase_price_outlier (filtro 'base'): 1,311,576 (87.0%)


## 2. Estructura relacional (Star Schema) para Tableau

Para que el equipo pueda **conectar relaciones en Tableau sin reprocesamiento manual**
y sin duplicar métricas, se construye un **modelo en estrella**:

- **`fact_transacciones`**: una fila por transacción válida. Contiene **únicamente
  las métricas numéricas** (`purchase_price`, `sqm_price`, `sqm_price_real`) y las
  claves foráneas (`tiempo_id`, `geografia_id`, `vivienda_id`, `macro_id`) + flags
  de calidad heredadas de TB2.
- **`dim_tiempo`**: una fila por fecha única. Atributos descriptivos/de segmentación
  (`year`, `quarter`, `periodo_macro`, `crisis_period`) — **sin métricas**.
- **`dim_geografia`**: una fila por combinación única `(zip_code, city, region)`.
  Incluye el flag de segmentación `es_capital` — **sin métricas**.
- **`dim_vivienda`**: una fila por combinación única `(house_type, sqm, no_rooms,
  year_build)`. Incluye `size_category`, `rooms_category`, `es_summerhouse`,
  `edad_vivienda` — **sin métricas**.
- **`dim_macro`**: una fila por año. Variables macro (tasas, inflación, bonos)
  — son **atributos del año**, no métricas de transacción, por lo que viven
  aquí y no se repiten en `fact_transacciones`.

**Regla de control de duplicidad**: ninguna columna de métrica (`purchase_price`,
`sqm_price`, `sqm_price_real`, conteos) aparece en más de una tabla. Las
dimensiones sólo contienen atributos descriptivos / llaves de segmentación.
Esto se valida explícitamente en la sección 3.

> **Adaptación a los valores reales** (única diferencia de lógica frente a la
> versión de desarrollo con muestra sintética):
> - `es_capital` ya no puede definirse como `region == "København"` porque en el
>   dataset real `region` son 4 agregados amplios (Zealand incluye tanto Copenhague
>   como zonas rurales). Se usa en su lugar `area == "Capital, Copenhagen"`
>   (verificado 1:1 con `zip_code`, sin ambigüedad), incorporando `area` a
>   `dim_geografia` sólo como insumo de este flag.
> - `es_summerhouse` se define como `house_type == "Summerhouse"` (equivalente real
>   a `"Fritidshus"` en la muestra sintética).


In [7]:
def clasif_periodo_macro(y: int) -> str:
    if y < 2000:
        return "pre_2000"
    elif y < 2008:
        return "boom"
    elif y < 2013:
        return "crisis"
    elif y < 2020:
        return "recuperacion"
    else:
        return "post_covid"


def build_dim_tiempo(df: pd.DataFrame) -> pd.DataFrame:
    dim = df[["date", "year", "quarter"]].drop_duplicates().copy()
    dim["periodo_macro"] = dim["year"].apply(clasif_periodo_macro)
    dim["crisis_period"] = dim["year"].isin(PARAMS["crisis_years"]).astype("int8")
    dim = dim.sort_values("date").reset_index(drop=True)
    dim.insert(0, "tiempo_id", dim.index + 1)
    return dim


def build_dim_geografia(df: pd.DataFrame) -> pd.DataFrame:
    dim = df[["zip_code", "city", "region", "area"]].drop_duplicates().reset_index(drop=True).copy()
    # es_capital: 'area' identifica Copenhague de forma 1:1 con zip_code (region no lo permite)
    dim["es_capital"] = (dim["area"] == "Capital, Copenhagen").astype("int8")
    dim = dim.drop(columns=["area"])
    dim.insert(0, "geografia_id", dim.index + 1)
    return dim


def build_dim_vivienda(df: pd.DataFrame) -> pd.DataFrame:
    dim = df[["house_type", "sqm", "no_rooms", "year_build"]].drop_duplicates().reset_index(drop=True).copy()
    dim["es_summerhouse"] = (dim["house_type"] == "Summerhouse").astype("int8")
    dim["rooms_category"] = pd.cut(
        dim["no_rooms"], bins=[-np.inf, 2, 4, 6, np.inf], labels=["Small", "Medium", "Large", "XLarge"]
    ).astype(str)
    sqm_q = dim["sqm"].quantile([0.33, 0.67])
    dim["size_category"] = pd.cut(
        dim["sqm"], bins=[-np.inf, sqm_q[0.33], sqm_q[0.67], np.inf], labels=["Small", "Medium", "Large"]
    ).astype(str)
    dim["edad_vivienda"] = (PARAMS["cpi_base_year"] - dim["year_build"]).clip(0, 300)
    dim.insert(0, "vivienda_id", dim.index + 1)
    return dim


def build_dim_macro(df: pd.DataFrame) -> pd.DataFrame:
    macro_cols = [c for c in ["nom_interest_rate_pct", "dk_ann_infl_rate_pct", "yield_mortgage_bonds_pct"] if c in df.columns]
    dim = df[["year"] + macro_cols + ["macro_nulo"]].drop_duplicates(subset=["year"]).copy()
    dim = dim.sort_values("year").reset_index(drop=True)
    dim.insert(0, "macro_id", dim.index + 1)
    return dim


def build_fact_transacciones(df, dim_tiempo, dim_geo, dim_viv, dim_macro) -> pd.DataFrame:
    fact = df.merge(dim_tiempo[["tiempo_id", "date"]], on="date", how="left")
    fact = fact.merge(dim_geo[["geografia_id", "zip_code", "city", "region"]], on=["zip_code", "city", "region"], how="left")
    fact = fact.merge(dim_viv[["vivienda_id", "house_type", "sqm", "no_rooms", "year_build"]], on=["house_type", "sqm", "no_rooms", "year_build"], how="left")
    fact = fact.merge(dim_macro[["macro_id", "year"]], on="year", how="left")

    metric_cols = ["purchase_price", "sqm_price", "sqm_price_real"]
    flag_cols = ["sales_type_valido", "purchase_price_outlier", "periodo_preliminar"]
    fk_cols = ["tiempo_id", "geografia_id", "vivienda_id", "macro_id"]

    fact_final = fact[fk_cols + metric_cols + flag_cols].copy()
    fact_final.insert(0, "transaction_id", range(1, len(fact_final) + 1))
    return fact_final


dim_tiempo = build_dim_tiempo(silver)
dim_geo    = build_dim_geografia(silver)
dim_viv    = build_dim_vivienda(silver)
dim_macro  = build_dim_macro(silver)
fact       = build_fact_transacciones(silver, dim_tiempo, dim_geo, dim_viv, dim_macro)

for name, t in [("fact_transacciones", fact), ("dim_tiempo", dim_tiempo),
                ("dim_geografia", dim_geo), ("dim_vivienda", dim_viv), ("dim_macro", dim_macro)]:
    print(f"{name:20s} {len(t):>9,} filas x {t.shape[1]} cols  -> {list(t.columns)}")

print()
print(f"Compresión dim_geografia vs fact: {len(dim_geo)/len(fact):.3%}  ({len(dim_geo):,} vs {len(fact):,})")
print(f"Compresión dim_vivienda  vs fact: {len(dim_viv)/len(fact):.3%}  ({len(dim_viv):,} vs {len(fact):,})")
print("Con el Silver real (zip_code ~ 941 códigos postales daneses, sqm/no_rooms/year_build")
print("con valores repetidos entre transacciones), dim_geografia comprime a ~0.06% del fact")
print("(965 filas vs 1.5M) y dim_vivienda a ~21.7% (327k combinaciones únicas de vivienda).")


fact_transacciones   1,507,908 filas x 11 cols  -> ['transaction_id', 'tiempo_id', 'geografia_id', 'vivienda_id', 'macro_id', 'purchase_price', 'sqm_price', 'sqm_price_real', 'sales_type_valido', 'purchase_price_outlier', 'periodo_preliminar']
dim_tiempo              11,742 filas x 6 cols  -> ['tiempo_id', 'date', 'year', 'quarter', 'periodo_macro', 'crisis_period']
dim_geografia              965 filas x 5 cols  -> ['geografia_id', 'zip_code', 'city', 'region', 'es_capital']
dim_vivienda           327,384 filas x 9 cols  -> ['vivienda_id', 'house_type', 'sqm', 'no_rooms', 'year_build', 'es_summerhouse', 'rooms_category', 'size_category', 'edad_vivienda']
dim_macro                   33 filas x 6 cols  -> ['macro_id', 'year', 'nom_interest_rate_pct', 'dk_ann_infl_rate_pct', 'yield_mortgage_bonds_pct', 'macro_nulo']

Compresión dim_geografia vs fact: 0.064%  (965 vs 1,507,908)
Compresión dim_vivienda  vs fact: 21.711%  (327,384 vs 1,507,908)
Con el Silver real (zip_code ~ 941 códigos post

## 3. Validación de la estructura relacional

Criterio de aprobación: *"la estructura relacional está validada y no duplica
métricas sin control"*. Se valida:

1. **Integridad referencial**: toda llave foránea en `fact_transacciones` existe
   en su tabla de dimensión correspondiente (sin huérfanos).
2. **Unicidad de llaves**: cada `*_id` es único en su tabla de dimensión.
3. **Sin duplicidad de métricas**: ninguna columna de `METRIC_COLS` aparece en
   más de una tabla del star schema.


In [8]:
METRIC_COLS = {"purchase_price", "sqm_price", "sqm_price_real"}

errors = []

# 1) Integridad referencial
checks_fk = {
    "tiempo_id": dim_tiempo["tiempo_id"],
    "geografia_id": dim_geo["geografia_id"],
    "vivienda_id": dim_viv["vivienda_id"],
    "macro_id": dim_macro["macro_id"],
}
for fk, valid_ids in checks_fk.items():
    orphans = fact[~fact[fk].isin(valid_ids) & fact[fk].notna()]
    pct_null = fact[fk].isna().mean() * 100
    print(f"  {fk:14s} -> huérfanos: {len(orphans):>6,} | nulos: {pct_null:5.2f}%")
    if len(orphans) > 0:
        errors.append(f"{fk}: {len(orphans)} huérfanos")

# 2) Unicidad de llaves de dimensión
for name, t, key in [("dim_tiempo", dim_tiempo, "tiempo_id"), ("dim_geografia", dim_geo, "geografia_id"),
                      ("dim_vivienda", dim_viv, "vivienda_id"), ("dim_macro", dim_macro, "macro_id")]:
    dup = t[key].duplicated().sum()
    print(f"  {name:14s} llave {key} duplicada: {dup}")
    if dup:
        errors.append(f"{name}: {dup} llaves duplicadas")

# 3) Sin duplicidad de métricas entre tablas
all_tables = {"fact_transacciones": fact, "dim_tiempo": dim_tiempo, "dim_geografia": dim_geo,
               "dim_vivienda": dim_viv, "dim_macro": dim_macro}
for metric in METRIC_COLS:
    tables_with_metric = [n for n, t in all_tables.items() if metric in t.columns]
    print(f"  métrica '{metric}' presente en: {tables_with_metric}")
    if len(tables_with_metric) != 1:
        errors.append(f"métrica '{metric}' duplicada en {tables_with_metric}")

print()
if errors:
    print("FALLÓ la validación relacional:")
    for e in errors:
        print(" -", e)
else:
    print("OK — estructura relacional validada: sin huérfanos, llaves únicas y métricas sin duplicar.")


  tiempo_id      -> huérfanos:      0 | nulos:  0.00%
  geografia_id   -> huérfanos:      0 | nulos:  0.00%
  vivienda_id    -> huérfanos:      0 | nulos:  0.00%
  macro_id       -> huérfanos:      0 | nulos:  0.00%
  dim_tiempo     llave tiempo_id duplicada: 0
  dim_geografia  llave geografia_id duplicada: 0
  dim_vivienda   llave vivienda_id duplicada: 0
  dim_macro      llave macro_id duplicada: 0
  métrica 'purchase_price' presente en: ['fact_transacciones']
  métrica 'sqm_price_real' presente en: ['fact_transacciones']
  métrica 'sqm_price' presente en: ['fact_transacciones']

OK — estructura relacional validada: sin huérfanos, llaves únicas y métricas sin duplicar.


## 4. Métricas derivadas para visualización

Cinco KPIs, cada uno calculado **a partir del star schema** (join `fact` + `dims`)
y exportado como un *mart* a un **grano explícito**. Documentar el grano evita que,
al combinarlos en un mismo dashboard de Tableau, se produzcan duplicaciones por
*data blending* (cada mart se usa como fuente independiente para su propia vista,
tal como define `docs/tableau-dashboard-design.md`).

| Mart | Grano (clave compuesta) | KPI | Vista Tableau |
|---|---|---|---|
| `mart_quarterly_regional_index` | `(region, quarter)` | Índice regional (base 1992=100) | Vista 1 — Línea de tiempo |
| `mart_drawdowns` | `(region, house_type, quarter)` | Drawdown pico-valle | Vista 3 — Crisis |
| `mart_volatility` | `(house_type, quarter)` | Volatilidad rolling 4Q | Vista 4 — Volatilidad |
| `mart_macro_correlation` | `(quarter)` | Elasticidad volumen-bonos (lag 2Q) | Vista 5 — Macro |
| `mart_transactions_map` | `(zip_code, year)` | Precio y volumen por zona | Vista 2 — Mapa |

Filtro común aplicado a todos los marts (excepto donde se indique): se excluyen
filas con `purchase_price_outlier = True` y `sales_type_valido = False` (heredado
de TB2 — *"preferimos flags sobre eliminación"*, las flags se preservan en
`fact_transacciones` pero los marts agregados las filtran).


In [9]:
# Vista enriquecida: fact + atributos de dimensiones necesarios para agregar
fact_enriched = (
    fact
    .merge(dim_tiempo[["tiempo_id", "year", "quarter", "periodo_macro", "crisis_period"]], on="tiempo_id", how="left")
    .merge(dim_geo[["geografia_id", "region", "es_capital"]], on="geografia_id", how="left")
    .merge(dim_viv[["vivienda_id", "house_type", "size_category", "es_summerhouse"]], on="vivienda_id", how="left")
    .merge(dim_macro[["macro_id", "yield_mortgage_bonds_pct"]], on="macro_id", how="left")
)

# Filtro estándar TB2: transacciones de mercado, sin outliers de precio
base = fact_enriched[
    fact_enriched["sales_type_valido"] & ~fact_enriched["purchase_price_outlier"]
].copy()

print(f"fact_enriched: {len(fact_enriched):,} filas")
print(f"base (filtrada para marts): {len(base):,} filas ({len(base)/len(fact_enriched):.1%})")


fact_enriched: 1,507,908 filas
base (filtrada para marts): 1,311,576 filas (87.0%)


In [10]:
# ── KPI 2: mart_quarterly_regional_index — grano (region, quarter) ──────────
q = (
    base.groupby(["region", "quarter"], observed=True)
    .agg(avg_sqm_price_real=("sqm_price_real", "mean"), n_transactions=("transaction_id", "count"))
    .reset_index()
    .sort_values(["region", "quarter"])
)

base_year = PARAMS["index_base_year"]
q["year"] = q["quarter"].str[:4].astype(int)
base_price = q[q["year"] == base_year].groupby("region")["avg_sqm_price_real"].mean().rename("base_year_price")
q = q.merge(base_price, on="region", how="left")
# Fallback si una región no tiene observaciones en el año base: usar su primer trimestre disponible
for reg in q.loc[q["base_year_price"].isna(), "region"].unique():
    q.loc[q["region"] == reg, "base_year_price"] = q.loc[q["region"] == reg, "avg_sqm_price_real"].iloc[0]

q["regional_index"] = (q["avg_sqm_price_real"] / q["base_year_price"] * 100).round(2)
mart_regional_index = q[["quarter", "region", "avg_sqm_price_real", "regional_index", "n_transactions"]].copy()
mart_regional_index["avg_sqm_price_real"] = mart_regional_index["avg_sqm_price_real"].round(1)

print(f"mart_quarterly_regional_index: {len(mart_regional_index):,} filas")
mart_regional_index.head()


mart_quarterly_regional_index: 504 filas


,quarter,region,avg_sqm_price_real,regional_index,n_transactions
0,1996Q1,Bornholm,8520.5,100.00,54
1,1996Q2,Bornholm,7140.4,83.80,55
2,1996Q3,Bornholm,10904.0,127.97,61
3,1996Q4,Bornholm,11702.2,137.34,56
4,1997Q1,Bornholm,9181.7,107.76,47


In [11]:
# ── KPI 3: mart_drawdowns — grano (region, house_type, quarter) ─────────────
dd = (
    base.groupby(["region", "house_type", "quarter"], observed=True)
    .agg(avg_sqm_price_real=("sqm_price_real", "mean"), n_transactions=("transaction_id", "count"))
    .reset_index()
    .sort_values(["region", "house_type", "quarter"])
)

n_before = len(dd)
dd = dd[dd["n_transactions"] >= PARAMS["drawdown_min_obs"]].copy()
print(f"drawdowns: filtro n_transactions >= {PARAMS['drawdown_min_obs']}: {n_before:,} -> {len(dd):,} filas")

dd["cumulative_max"] = dd.groupby(["region", "house_type"], observed=True)["avg_sqm_price_real"].cummax()
dd["drawdown_pct"] = ((dd["avg_sqm_price_real"] - dd["cumulative_max"]) / dd["cumulative_max"] * 100).round(2)
dd[["avg_sqm_price_real", "cumulative_max"]] = dd[["avg_sqm_price_real", "cumulative_max"]].round(1)

mart_drawdowns = dd[["quarter", "region", "house_type", "avg_sqm_price_real", "cumulative_max", "drawdown_pct", "n_transactions"]].copy()
print(f"mart_drawdowns: {len(mart_drawdowns):,} filas")
mart_drawdowns.sort_values("drawdown_pct").head()


drawdowns: filtro n_transactions >= 5: 2,503 -> 2,423 filas
mart_drawdowns: 2,423 filas


,quarter,region,house_type,avg_sqm_price_real,cumulative_max,drawdown_pct,n_transactions
76,2018Q3,Bornholm,Apartment,7844.9,99418.3,-92.11,5
69,2016Q2,Bornholm,Apartment,10795.9,99418.3,-89.14,6
67,2015Q4,Bornholm,Apartment,11993.8,99418.3,-87.94,8
60,2014Q1,Bornholm,Apartment,12458.8,99418.3,-87.47,5
97,2024Q1,Bornholm,Apartment,12762.0,99418.3,-87.16,10


In [12]:
# ── KPI 4: mart_volatility — grano (house_type, quarter) ────────────────────
vol = (
    base.groupby(["house_type", "quarter"], observed=True)
    .agg(avg_sqm_price_real=("sqm_price_real", "mean"))
    .reset_index()
    .sort_values(["house_type", "quarter"])
)

vol["pct_change"] = vol.groupby("house_type", observed=True)["avg_sqm_price_real"].pct_change()
w = PARAMS["volatility_window_q"]
vol["volatility_4q"] = (
    vol.groupby("house_type", observed=True)["pct_change"]
    .transform(lambda x: x.rolling(w, min_periods=w).std())
)
vol["avg_sqm_price_real"] = vol["avg_sqm_price_real"].round(1)
vol["volatility_4q"] = vol["volatility_4q"].round(4)

mart_volatility = vol[["quarter", "house_type", "avg_sqm_price_real", "volatility_4q"]].copy()
print(f"mart_volatility: {len(mart_volatility):,} filas")
mart_volatility.dropna().head()


mart_volatility: 650 filas


,quarter,house_type,avg_sqm_price_real,volatility_4q
4,1993Q1,Apartment,36995.8,0.1890
5,1993Q2,Apartment,43167.8,0.1505
6,1993Q3,Apartment,29609.6,0.2097
7,1993Q4,Apartment,33669.0,0.2225
8,1994Q1,Apartment,31536.5,0.2221


In [13]:
# ── KPI 5: mart_macro_correlation — grano (quarter) ─────────────────────────
# Nota: usa fact_enriched (NO 'base') porque las transacciones excluidas por
# sales_type/outlier siguen contando para el VOLUMEN de mercado y la serie macro
# es por año/trimestre, no por transacción.
mc = (
    fact_enriched.groupby("quarter")
    .agg(n_transactions=("transaction_id", "count"), avg_bond_yield=("yield_mortgage_bonds_pct", "mean"))
    .reset_index()
    .sort_values("quarter")
)

lag = PARAMS["macro_lag_quarters"]
mc["bond_yield_lag"] = mc["avg_bond_yield"].shift(lag)
mc["volume_bond_corr_8q"] = mc["n_transactions"].rolling(8, min_periods=4).corr(mc["bond_yield_lag"])
mc["avg_bond_yield"] = mc["avg_bond_yield"].round(2)
mc["bond_yield_lag"] = mc["bond_yield_lag"].round(2)
mc["volume_bond_corr_8q"] = mc["volume_bond_corr_8q"].round(3)

mart_macro_correlation = mc.copy()
print(f"mart_macro_correlation: {len(mart_macro_correlation):,} filas")
mart_macro_correlation.dropna().head()


mart_macro_correlation: 130 filas


,quarter,n_transactions,avg_bond_yield,bond_yield_lag,volume_bond_corr_8q
5,1993Q2,4687,8.16,10.14,0.870
6,1993Q3,5174,8.16,8.16,-0.745
7,1993Q4,4959,8.16,8.16,-0.792
8,1994Q1,4758,8.39,8.16,-0.792
9,1994Q2,5894,8.39,8.16,-0.805


In [14]:
# ── mart_transactions_map — grano (zip_code, year) ──────────────────────────
fm = base.merge(dim_geo[["geografia_id", "zip_code", "city"]], on="geografia_id", how="left")
fm["year"] = fm["quarter"].str[:4].astype(int)

mart_transactions_map = (
    fm.groupby(["year", "zip_code"], observed=True)
    .agg(
        avg_sqm_price_real=("sqm_price_real", "mean"),
        n_transactions=("transaction_id", "count"),
        city=("city", lambda x: x.mode().iloc[0] if len(x) else "Unknown"),
        region=("region", lambda x: x.mode().iloc[0] if len(x) else "Unknown"),
    )
    .reset_index()
)
mart_transactions_map["avg_sqm_price_real"] = mart_transactions_map["avg_sqm_price_real"].round(1)

print(f"mart_transactions_map: {len(mart_transactions_map):,} filas")
mart_transactions_map.head()


mart_transactions_map: 24,067 filas


,year,zip_code,avg_sqm_price_real,n_transactions,city,region
0,1992,1051,9936.1,1,København K,Zealand
1,1992,1052,45949.5,12,København K,Zealand
2,1992,1112,17933.4,1,København K,Zealand
3,1992,1127,24186.5,1,København K,Zealand
4,1992,1201,13129.8,1,København K,Zealand


## 5. Segmentación

Se definen **cuatro segmentos**, cada uno alimentado por un atributo ya presente
en una dimensión del star schema (sección 2), por lo que **no requieren nuevas
columnas de métricas** — sólo recortes/`GROUP BY` sobre los marts existentes.
Cada segmento responde directamente a una de las hipótesis de trabajo del
proyecto.

| # | Segmento | Definición (datos reales) | Tabla / columna | Hipótesis relacionada |
|---|---|---|---|---|
| 1 | **Capital vs. Provincias** | `es_capital = 1` si `area = "Capital, Copenhagen"`, 0 en otro caso | `dim_geografia.es_capital` | H2 — Efecto Capital (resiliencia ante shocks) |
| 2 | **Régimen macroeconómico** | `periodo_macro ∈ {pre_2000, boom, crisis, recuperacion, post_covid}` por año | `dim_tiempo.periodo_macro` | Pregunta principal — "distintos regímenes de tasas/inflación" |
| 3 | **Tipología: vivienda permanente vs. segunda vivienda** | `es_summerhouse = 1` si `house_type = "Summerhouse"` | `dim_vivienda.es_summerhouse` | H3 — Volatilidad por tipología |
| 4 | **Tamaño de vivienda** | `size_category ∈ {Small, Medium, Large}` (terciles de `sqm`) | `dim_vivienda.size_category` | Análisis exploratorio adicional (no ligado a hipótesis principal, soporte para Vista 4) |

A continuación se construye `mart_segment_summary`, un mart adicional a **grano
`(quarter, segmento_geografico, periodo_macro, tipologia_segmento)`** pensado
para los filtros cruzados del dashboard (p. ej. comparar Capital vs. Provincias
*dentro* de cada régimen macroeconómico, algo que los 5 marts de TB3 no permiten
de forma directa porque cada uno tiene un grano fijo distinto).


In [15]:
seg = base.copy()
seg["segmento_geografico"] = np.where(seg["es_capital"] == 1, "Capital", "Provincias")
seg["tipologia_segmento"] = np.where(seg["es_summerhouse"] == 1, "Segunda vivienda", "Vivienda permanente")

mart_segment_summary = (
    seg.groupby(["quarter", "segmento_geografico", "periodo_macro", "tipologia_segmento"], observed=True)
    .agg(
        avg_sqm_price_real=("sqm_price_real", "mean"),
        median_sqm_price_real=("sqm_price_real", "median"),
        n_transactions=("transaction_id", "count"),
    )
    .reset_index()
)
mart_segment_summary[["avg_sqm_price_real", "median_sqm_price_real"]] = (
    mart_segment_summary[["avg_sqm_price_real", "median_sqm_price_real"]].round(1)
)

print(f"mart_segment_summary: {len(mart_segment_summary):,} filas")
print()
print("Tamaño de cada segmento sobre 'base':")
print(f"  Capital:             {(base['es_capital']==1).sum():>9,} filas")
print(f"  Provincias:          {(base['es_capital']==0).sum():>9,} filas")
print(f"  Vivienda permanente: {(base['es_summerhouse']==0).sum():>9,} filas")
print(f"  Segunda vivienda:    {(base['es_summerhouse']==1).sum():>9,} filas")
mart_segment_summary.head()


mart_segment_summary: 510 filas

Tamaño de cada segmento sobre 'base':
  Capital:               205,048 filas
  Provincias:          1,106,528 filas
  Vivienda permanente: 1,174,713 filas
  Segunda vivienda:      136,863 filas


,quarter,segmento_geografico,periodo_macro,tipologia_segmento,avg_sqm_price_real,median_sqm_price_real,n_transactions
0,1992Q1,Capital,pre_2000,Vivienda permanente,17700.8,12630.2,530
1,1992Q1,Provincias,pre_2000,Segunda vivienda,10242.8,9042.6,121
2,1992Q1,Provincias,pre_2000,Vivienda permanente,14601.2,8206.1,2960
3,1992Q2,Capital,pre_2000,Vivienda permanente,14967.2,12379.9,600
4,1992Q2,Provincias,pre_2000,Segunda vivienda,10164.7,8706.8,216


### Lectura del segmento — ¿cómo afecta cada cálculo a la interpretación?

- **Capital vs. Provincias** (`es_capital`): al fijar este flag en `dim_geografia`
  (atributo, no agregación) a partir de `area == "Capital, Copenhagen"`, Tableau
  puede comparar `mart_regional_index` y `mart_drawdowns` filtrando por región sin
  recalcular nada — el índice y el drawdown de cada región ya están normalizados a
  su **propia base 1992**, por lo que la comparación Capital vs. Provincias es de
  *trayectoria relativa*, no de nivel absoluto de precios. Con datos reales,
  **Capital concentra 205,048 transacciones (15.6% de `base`)** frente a
  **1,106,528 en Provincias (84.4%)** — coherente con que Copenhague es una
  fracción minoritaria pero analíticamente central del mercado total.
- **Régimen macroeconómico** (`periodo_macro`): al ser una propiedad del **año**
  (vía `dim_tiempo`), permite cortar cualquier mart por régimen sin tocar el
  cálculo de la métrica. Importante: los regímenes son **anuales**, mientras que
  los marts son **trimestrales** — un mismo año aporta hasta 4 trimestres al
  mismo régimen, lo cual es correcto pero debe tenerse en cuenta al promediar
  (no ponderar por régimen sin considerar el desbalance de trimestres).
- **Tipología (segunda vivienda vs. permanente)**: `es_summerhouse` aísla
  `Summerhouse`. En `mart_volatility` y `mart_drawdowns`, comparar esta
  bandera permite leer directamente si "Summerhouse" (segunda vivienda,
  consumo discrecional) cae más que "Villa/Apartment" en crisis — la
  hipótesis H3 predice mayor volatilidad y drawdown en este segmento porque
  la demanda de segunda vivienda es más sensible al ciclo económico. Con
  datos reales, **Segunda vivienda representa 136,863 filas (10.4%)** frente a
  **1,174,713 de Vivienda permanente (89.6%)**.
- **Tamaño de vivienda** (`size_category`): terciles calculados sobre `sqm`
  de **todo el dataset** (no por región/año), por lo que es un corte relativo
  global. Útil como variable de control en Vista 4, pero **no** se usa como
  eje principal de ninguna hipótesis — se documenta para que el equipo no la
  interprete como comparable entre regímenes (los terciles no se recalculan
  por período).


## 6. Parámetros y lógica analítica — sensibilidad

`PARAMS` (sección 0) centraliza siete decisiones analíticas. Para que el equipo
pueda **explicar cómo cada cálculo afecta la interpretación**, se muestra a
continuación un ejemplo de sensibilidad para el parámetro más delicado:
`drawdown_min_obs`.


In [16]:
# Sensibilidad: drawdown_min_obs = 1 (sin filtro) vs. PARAMS (= 5)
dd_raw = (
    base.groupby(["region", "house_type", "quarter"], observed=True)
    .agg(avg_sqm_price_real=("sqm_price_real", "mean"), n_transactions=("transaction_id", "count"))
    .reset_index()
    .sort_values(["region", "house_type", "quarter"])
)
dd_raw["cumulative_max"] = dd_raw.groupby(["region", "house_type"], observed=True)["avg_sqm_price_real"].cummax()
dd_raw["drawdown_pct"] = (dd_raw["avg_sqm_price_real"] - dd_raw["cumulative_max"]) / dd_raw["cumulative_max"] * 100

low_obs = dd_raw[dd_raw["n_transactions"] < PARAMS["drawdown_min_obs"]]
worst_overall = dd_raw.loc[dd_raw["drawdown_pct"].idxmin()]
worst_low = low_obs.loc[low_obs["drawdown_pct"].idxmin()]
worst_kept = mart_drawdowns.loc[mart_drawdowns["drawdown_pct"].idxmin()]

print(f"Celdas totales (region, house_type, quarter) sin filtrar:          {len(dd_raw):>6,}")
print(f"Celdas con n_transactions < {PARAMS['drawdown_min_obs']} (descartadas por el parámetro): {len(low_obs):>6,}")
print()
print(f"Drawdown mínimo SIN filtro de muestra mínima (min_obs=1): {dd_raw['drawdown_pct'].min():.2f}%  "
      f"({worst_overall['region']} / {worst_overall['house_type']} / {worst_overall['quarter']}, "
      f"n={int(worst_overall['n_transactions'])})")
print(f"Drawdown mínimo CON drawdown_min_obs={PARAMS['drawdown_min_obs']}:           {mart_drawdowns['drawdown_pct'].min():.2f}%  "
      f"({worst_kept['region']} / {worst_kept['house_type']} / {worst_kept['quarter']}, "
      f"n={int(worst_kept['n_transactions'])})")
print(f"Drawdown mínimo SOLO en celdas descartadas (n<{PARAMS['drawdown_min_obs']}): {worst_low['drawdown_pct']:.2f}% "
      f"({worst_low['region']} / {worst_low['house_type']} / {worst_low['quarter']}, "
      f"n_transactions={int(worst_low['n_transactions'])})")
print()
print("Interpretación: las celdas (región, tipología, trimestre) con muy pocas")
print("transacciones son las que producen los drawdowns más extremos -un único")
print("registro atípico domina el promedio del trimestre-. El parámetro")
print("drawdown_min_obs evita confundir 'ruido de muestra pequeña' con una caída")
print("real de mercado durante una crisis. Con datos reales, el caso más extremo")
print("descartado (n<5) es Bornholm/Apartment en 2012Q3 (n=2, -97.25%); el peor")
print("drawdown que SÍ queda en el mart (n>=5) es Bornholm/Apartment en 2018Q3")
print("(-92.11%, n=5) — un mercado real, pequeño y poco líquido (apartamentos en")
print("Bornholm), consistente con la hipótesis de mayor volatilidad en segmentos")
print("de baja transaccionalidad.")


Celdas totales (region, house_type, quarter) sin filtrar:           2,503
Celdas con n_transactions < 5 (descartadas por el parámetro):     80

Drawdown mínimo SIN filtro de muestra mínima (min_obs=1): -97.25%  (Bornholm / Apartment / 2012Q3, n=2)
Drawdown mínimo CON drawdown_min_obs=5:           -92.11%  (Bornholm / Apartment / 2018Q3, n=5)
Drawdown mínimo SOLO en celdas descartadas (n<5): -97.25% (Bornholm / Apartment / 2012Q3, n_transactions=2)

Interpretación: las celdas (región, tipología, trimestre) con muy pocas
transacciones son las que producen los drawdowns más extremos -un único
registro atípico domina el promedio del trimestre-. El parámetro
drawdown_min_obs evita confundir 'ruido de muestra pequeña' con una caída
real de mercado durante una crisis. Con datos reales, el caso más extremo
descartado (n<5) es Bornholm/Apartment en 2012Q3 (n=2, -97.25%); el peor
drawdown que SÍ queda en el mart (n>=5) es Bornholm/Apartment en 2018Q3
(-92.11%, n=5) — un mercado real, pequeño y p

### Tabla de parámetros y su efecto en la interpretación

| Parámetro | Valor | Afecta a | Efecto en la interpretación si cambia |
|---|---|---|---|
| `cpi_base_year` | 2024 | `sqm_price_real` (todos los marts, ya calculado en Silver real) | Cambiar el año base reescala *todos* los precios reales por el mismo factor constante — no afecta comparaciones relativas (índices, drawdowns, volatilidad), sólo el nivel absoluto en DKK mostrado en tooltips. |
| `cpi_annual_rate` | 0.02 (legado / documental) | No se usa para recalcular `sqm_price_real` en esta corrida — el Silver real ya trae el deflactor aplicado con la serie real `dk_ann_infl_rate_pct` (compuesta hasta 2024). Se conserva en `PARAMS` como referencia del supuesto original y como fallback si se regenerase Silver sin esa serie. |
| `index_base_year` | 1992 | `mart_quarterly_regional_index` | Es el "100" de referencia. Si una región no tiene datos en 1992 (no ocurre con datos reales: las 4 regiones tienen observaciones desde 1992), se usaría su primer trimestre disponible como *fallback* — esto haría que esa región no fuera directamente comparable en nivel con las demás en los primeros años. |
| `crisis_years` | 2007-2012, 2020-2021 | `dim_tiempo.crisis_period`, sombreado en Vista 3 | Ampliar/recortar este rango cambia qué trimestres se resaltan como "crisis" y qué picos se usan como referencia de drawdown — define el rango "pico-valle-recuperación" que se analiza. |
| `drawdown_min_obs` | 5 | `mart_drawdowns` | Ver celda de sensibilidad arriba — valores muy bajos (1-2) introducen drawdowns espurios de hasta -97% (Bornholm/Apartment, n=2); valores muy altos eliminan celdas legítimas en regiones/tipologías poco transadas (p. ej. Summerhouse en Bornholm). |
| `volatility_window_q` | 4 | `mart_volatility` | Ventanas más cortas (1-2Q) son más ruidosas y reaccionan más rápido a shocks puntuales; ventanas más largas (8Q+) suavizan la serie pero retrasan la detección del inicio de una crisis. 4Q (1 año) es el estándar para "volatilidad anualizada". |
| `macro_lag_quarters` | 2 | `mart_macro_correlation` | Es la hipótesis H1 (rezago de 1-2 trimestres entre subida de tasas y caída de volumen). Cambiarlo a 0 testea correlación contemporánea (más débil esperada); a 4+ testea efectos de mediano plazo. |
| `iqr_multiplier` | 3.0 (heredado P5 de TB2) | `purchase_price_outlier`, por tanto **todos** los marts vía `base` (excluye 24,282 filas, 1.6% del Silver) | Un multiplicador más bajo (1.5, estándar Tukey) excluiría más transacciones "premium" legítimas como outliers; 3.0 es deliberadamente conservador para no perder señal en el segmento de vivienda de lujo. |

### Mapeo a parámetros interactivos de Tableau

Los parámetros anteriores son de **preparación de datos** (se fijan antes de
exportar los CSV; cambiarlos requiere re-ejecutar este notebook). Adicionalmente,
se definen **parámetros nativos de Tableau** —interactivos, sin reprocesamiento—
sobre los datos ya exportados:

- `p_year_range` (rango 1992–2024): filtra el eje temporal en las 5 vistas.
- `p_region` (multiselección sobre `dim_geografia.region`, incluye atajo
  "Capital" / "Provincias" vía `es_capital`).
- `p_house_type` (multiselección sobre `dim_vivienda.house_type`).
- `p_periodo_macro` (selección sobre `dim_tiempo.periodo_macro`) — para
  comparar el segmento 2 directamente en el dashboard.


## 7. Validación de consistencia

Antes de exportar, se verifica que:

1. Cada mart respeta su grano declarado (sin filas duplicadas por clave compuesta).
2. El total de `n_transactions` agregado en `mart_regional_index` no excede el
   total de filas en `base` (no se están "inflando" conteos por joins).
3. No hay valores nulos en las columnas llave de cada mart.


In [17]:
def check_grain(name, df, keys):
    dup = df.duplicated(subset=keys).sum()
    nulls = df[keys].isna().any().any()
    status = "OK" if dup == 0 and not nulls else "FALLA"
    print(f"  [{status}] {name:30s} grano={keys} | duplicados={dup} | nulos_en_llave={nulls}")
    return dup == 0 and not nulls


checks = []
checks.append(check_grain("mart_quarterly_regional_index", mart_regional_index, ["region", "quarter"]))
checks.append(check_grain("mart_drawdowns", mart_drawdowns, ["region", "house_type", "quarter"]))
checks.append(check_grain("mart_volatility", mart_volatility, ["house_type", "quarter"]))
checks.append(check_grain("mart_macro_correlation", mart_macro_correlation, ["quarter"]))
checks.append(check_grain("mart_transactions_map", mart_transactions_map, ["zip_code", "year"]))
checks.append(check_grain("mart_segment_summary", mart_segment_summary,
                           ["quarter", "segmento_geografico", "periodo_macro", "tipologia_segmento"]))

# Chequeo de conteo: la suma de n_transactions de mart_regional_index debe ser
# exactamente igual al tamaño de `base` (mismo grano de filtro, sin doble conteo)
total_base = len(base)
total_regional = mart_regional_index["n_transactions"].sum()
total_map = mart_transactions_map["n_transactions"].sum()
print()
print(f"filas en 'base' (filtro estándar):              {total_base:>10,}")
print(f"suma n_transactions mart_regional_index:        {total_regional:>10,}")
print(f"suma n_transactions mart_transactions_map:      {total_map:>10,}")

assert total_regional == total_base, "mart_regional_index no conserva el total de transacciones"
assert total_map == total_base, "mart_transactions_map no conserva el total de transacciones"
print()
print("OK — totales consistentes, sin doble conteo." if all(checks) else "Revisar checks de grano arriba.")


  [OK] mart_quarterly_regional_index  grano=['region', 'quarter'] | duplicados=0 | nulos_en_llave=False
  [OK] mart_drawdowns                 grano=['region', 'house_type', 'quarter'] | duplicados=0 | nulos_en_llave=False
  [OK] mart_volatility                grano=['house_type', 'quarter'] | duplicados=0 | nulos_en_llave=False
  [OK] mart_macro_correlation         grano=['quarter'] | duplicados=0 | nulos_en_llave=False
  [OK] mart_transactions_map          grano=['zip_code', 'year'] | duplicados=0 | nulos_en_llave=False
  [OK] mart_segment_summary           grano=['quarter', 'segmento_geografico', 'periodo_macro', 'tipologia_segmento'] | duplicados=0 | nulos_en_llave=False

filas en 'base' (filtro estándar):               1,311,576
suma n_transactions mart_regional_index:         1,311,576
suma n_transactions mart_transactions_map:       1,311,576

OK — totales consistentes, sin doble conteo.


## 8. Exportación de fuentes finales para Tableau

Se exportan dos grupos de archivos a `data/processed/`:

- **`star_schema/`**: las 5 tablas del modelo relacional (`fact_transacciones` +
  4 dimensiones). En Tableau se conectan vía **relaciones** (no joins físicos)
  usando las llaves `*_id` — esto es lo que permite analizar a distintos grados
  de granularidad sin duplicar filas de hechos.
- **`tableau_marts/`**: los 5 marts de KPIs (TB3) + el nuevo `mart_segment_summary`
  (TB4), cada uno pre-agregado a su grano documentado, listo para usarse como
  fuente independiente por vista.

Todas las columnas quedan con tipos nativos (sin objetos `Period`/`category` que
Tableau no reconozca) y sin índices adicionales — **se pueden abrir directamente
en Tableau Desktop con "Conectar a archivo de texto" sin pasos manuales**.


In [18]:
# Star schema
star_tables = {
    "fact_transacciones": fact,
    "dim_tiempo": dim_tiempo,
    "dim_geografia": dim_geo,
    "dim_vivienda": dim_viv,
    "dim_macro": dim_macro,
}
for name, t in star_tables.items():
    out_path = OUT_STAR / f"{name}.csv"
    t.to_csv(out_path, index=False)
    print(f"  {out_path}  ({len(t):,} filas x {t.shape[1]} cols)")

print()

# Marts de métricas
mart_tables = {
    "mart_quarterly_regional_index": mart_regional_index,
    "mart_drawdowns": mart_drawdowns,
    "mart_volatility": mart_volatility,
    "mart_macro_correlation": mart_macro_correlation,
    "mart_transactions_map": mart_transactions_map,
    "mart_segment_summary": mart_segment_summary,
}
for name, t in mart_tables.items():
    out_path = OUT_MARTS / f"{name}.csv"
    t.to_csv(out_path, index=False)
    print(f"  {out_path}  ({len(t):,} filas x {t.shape[1]} cols)")


  /data/processed/star_schema/fact_transacciones.csv  (1,507,908 filas x 11 cols)
  /data/processed/star_schema/dim_tiempo.csv  (11,742 filas x 6 cols)
  /data/processed/star_schema/dim_geografia.csv  (965 filas x 5 cols)
  /data/processed/star_schema/dim_vivienda.csv  (327,384 filas x 9 cols)
  /data/processed/star_schema/dim_macro.csv  (33 filas x 6 cols)

  /data/processed/tableau_marts/mart_quarterly_regional_index.csv  (504 filas x 5 cols)
  /data/processed/tableau_marts/mart_drawdowns.csv  (2,423 filas x 7 cols)
  /data/processed/tableau_marts/mart_volatility.csv  (650 filas x 4 cols)
  /data/processed/tableau_marts/mart_macro_correlation.csv  (130 filas x 5 cols)
  /data/processed/tableau_marts/mart_transactions_map.csv  (24,067 filas x 6 cols)
  /data/processed/tableau_marts/mart_segment_summary.csv  (510 filas x 7 cols)


In [19]:
# Resumen final de esquema (para el documento de reglas / README de tableau/)
schema_summary = []
for name, t in {**star_tables, **mart_tables}.items():
    for col in t.columns:
        schema_summary.append({"tabla": name, "columna": col, "dtype": str(t[col].dtype)})

schema_df = pd.DataFrame(schema_summary)
schema_path = OUT_MARTS.parent / "tableau_sources_schema.csv"
schema_df.to_csv(schema_path, index=False)
print(f"Esquema completo de fuentes -> {schema_path} ({len(schema_df)} columnas en {len(star_tables)+len(mart_tables)} tablas)")
schema_df.groupby("tabla").size()


Esquema completo de fuentes -> /data/processed/tableau_sources_schema.csv (71 columnas en 11 tablas)


,0
tabla,
dim_geografia,5
dim_macro,6
dim_tiempo,6
dim_vivienda,9
fact_transacciones,11
mart_drawdowns,7
mart_macro_correlation,5
mart_quarterly_regional_index,5
mart_segment_summary,7
